In [1]:
from transformers import Qwen2VLModel, AutoTokenizer, AutoModelForVision2Seq, AutoProcessor
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch

#model = Qwen2VLModel.from_pretrained("qwen/Qwen2-VL-2B-Instruct", trust_remote_code=True, device_map="auto")
# model = AutoModelForVision2Seq.from_pretrained("qwen/Qwen2-VL-2B-Instruct-AWQ", trust_remote_code=True, device_map="auto",from_tf=True)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct-AWQ", torch_dtype="auto", device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("qwen/Qwen2-VL-2B-Instruct-AWQ", trust_remote_code=True)

/home/can/miniconda3/envs/vila/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2025-09-03 17:26:02,023] [INFO] [real_accelerator.py:110:get_accelerator] Setting ds_accelerator to cuda (auto detect)


We suggest you to set `torch_dtype=torch.float16` for better efficiency with AWQ.
/home/can/miniconda3/envs/vila/lib/python3.10/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/home/can/miniconda3/envs/vila/lib/python3.10/site-packages/accelerate/utils/modeling.py:1241: DeprecationWarning: The 'warn' 

In [2]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(
    "qwen/Qwen2-VL-2B-Instruct", 
)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [3]:
import json
with open("/media/data/nuplanqa/nuplan_train_v2.json", "r") as f:
    data = json.load(f)


In [4]:
question = data[0]['QA'][0]['q']

In [5]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "/media/data/nuplanqa/nuplan-v1.1_train_camera_0/nuplan-v1.1_train_camera_0/0a0a7f3bd0255653.jpg",
            },
            {"type": "text", "text": question },
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
from qwen_vl_utils import process_vision_info
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

/home/can/miniconda3/envs/vila/lib/python3.10/site-packages/torch/nn/modules/conv.py:605: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv3d(


AttributeError: module 'triton.language' has no attribute 'interleave'